# 1. CLASS-4 LOSS RESHAPING — AeroSync-Mamba v20

v20 starts from v18. It tests two replacements for the classification loss L_cls, aimed at **Class 4 recall** and
at **adjacent-severity errors**. It uses a paired 5-seed design from the start.

**Why a replacement, not an extra term.** v14 added an ordinal MSE term on top of cross-entropy. It hurt macro-F1
and *increased* adjacent errors (94 → 113) at every weight tried. Two likely causes: the extra objective works
against CE at the exact boundary that matters, and the two terms were on very different scales. Both candidates
here **reshape L_cls itself**. λ_align = λ_cons = 0.1 and the InfoNCE and KL terms stay exactly as before.

| | L_cls (per example i, true class y) |
|---|---|
| **Control** (v16–v18) | α_y · (−log p_y), via `nn.CrossEntropyLoss(weight=α)` |
| **A: weighted focal** | α_y · (1 − p_t)^γ · (−log p_t), γ = 2.0, where p_t = p_y |
| **B: ordinal label smoothing** | α_y · (−Σ_k q(k \| y) log p(k)), q(k \| y) ∝ exp(−\|k − y\| / τ), τ = 1.0 |

Both candidates use the **same α_y** (inverse training-set class frequency, from the existing `train_class_weights`).
They also use the **same normalization** as `nn.CrossEntropyLoss(weight=α)`: Σ α_y · loss_i / Σ α_y. So with γ = 0,
or τ → 0, each candidate reduces exactly to the control's L_cls. Section 5 checks this on real data. Any difference
therefore comes from the reshaping, not from a change of scale. Each epoch also prints the weighted-CE value on the
same logits, to make any scale shift visible.

**Architecture.** v18 with **β hard-fixed at exactly 0.0**. The β question is parked, so this experiment is fully
separate from it. There is no warm-up and no learnable β. The v18 LayerNorm and zero-init of g stay, but at β = 0
they cannot reach the output. The lag term is asserted to be exactly 0 at every epoch.

**Control: v18's five β = 0.0 runs, reused, not retrained.** v18 = v20 at β = 0 with the CE loss.
- v18's CSV holds only the 4 aggregate metrics, so the control's confusion matrices are rebuilt from **v18's saved
  test probabilities** (`v18_test_probs/beta0.0_seed*_test_probs.npy`). To use them, attach v18's notebook output as
  a Kaggle input. The rebuilt metrics must equal the v18 values embedded in Section 2 exactly.
- If those files are not attached, the control is **regenerated** by default (5 extra runs,
  `REGENERATE_CONTROL_IF_MISSING`). Each regenerated run must reproduce v18's 4 metrics, so it is the same control
  reproduced, not a new one.

**Runs.** 10 new runs: candidate A on seeds 42, 123, 2024, 7, 99, then candidate B on the same seeds. Every run
records its full 5×5 confusion matrix, Class 4 recall, adjacent errors (|pred − true| = 1) and distant errors (≥ 2),
using the same split as v14, plus the 4 standard metrics.

**Two corrections to the premise, from the data itself:**
- **Class 4 is not the smallest training class.** Training windows per class are 906 / 1020 / 1004 / 1008 / 996, so
  class 0 is smallest and **α₄ ≈ 0.99**. Class weighting therefore gives Class 4 essentially no extra weight, and
  any Class 4 gain has to come from the reshaping itself. Class 4 *is* the smallest **test** class (201 windows), so
  one window is 0.005 of Class 4 recall.
- **Class 2 may be the worse class.** In v14's seed-42 baseline, recall was class 2 0.824 and class 4 0.846. Section
  7 therefore reports recall for every class, not only class 4.

| Decision | Value | Evidence status |
|---|---|---|
| Timestamps into HTT / LACA | Real per-window audio and IMU timestamps (not a synthetic `linspace`) | CORRECTNESS FIX (earlier versions passed an identical synthetic grid for every window) |
| Fusion backbone | Real `mamba_ssm.Mamba` selective-scan SSM; pure-PyTorch fallback only when no CUDA GPU is available (printed unmissably) | CORRECTNESS FIX (the earlier gated-conv block was not an SSM) |
| Audio bin selection | Strongest-N FFT bins by mean magnitude, computed once from **training** audio only, then frozen and reused for val/test | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): strongest-64 0.8736 vs first-64 0.8494, +0.0242 (bin-selection study) |
| Frequency budget N | 64 bins | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v6: N=64 0.8966 vs N=48 0.8863, **+0.0103**, which is below 0.0160. The other budgets clear it (N=32 +0.0249, N=128 +0.0166) except N=96 (+0.0142). |
| Audio compression | `sqrt(|rFFT|)` | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 sqrt 0.8966 vs log_db 0.8696 (+0.0270) and log1p 0.8649 (+0.0317) |
| Audio windowing | None; Hann windowing rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 no window 0.8649 vs Hann 0.8290 (+0.0359) |
| Audio normalization | None; z-score normalization rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 none 0.8966 vs z-score 0.8684 (+0.0282) |
| Audio time positions Ta | 64 per 2.56 s window | PROVISIONAL: v7 Ta=128 0.8967 vs Ta=64 0.8966 (−0.0001, a tie). The cheaper Ta=64 is kept. |
| Sensor input | Raw 6 channels (accel x/y/z, gyro x/y/z), Ts=128; derived magnitude features rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v8 raw6 0.8966 vs best derived set 0.8740 (+0.0226) |
| LACA temporal-bias scale β | **hard-fixed at 0.0 in v20** (the β question is parked) | NOT ESTABLISHED: v17 +0.0008 ± 0.0020 (3/5), v18 −0.0094 ± 0.0136 (1/5), v19 learnable β. Decoupled from this experiment. |
| LACA lag-bias form g(Δτ) | mlp `Linear(1,32) → Tanh → Linear(32,1)`, 97 params. **v18:** final Linear zero-initialized, output LayerNorm over each window's (T_m, T_n) grid (0 params) | PROVISIONAL: v10 mlp vs rbf +0.0029, vs bucketed +0.0030 (below 0.0160). The v18 stabilization is under test. |
| LACA heads | 4 | PROVISIONAL: v11 h=1 was +0.0008 (one test window) and h=4 was kept. h=2/8/16 were 0.006–0.012 worse (below 0.0160). |
| Mamba fusion | 2 blocks, d_state=16, d_conv=2 (expand 2) | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v12: 2 vs 4 blocks +0.0023, d_state 16 vs 8 +0.0172, d_conv 2 vs 4 +0.0059. |
| λ_align (InfoNCE) | 0.1 | PROVISIONAL: v13 removing L_align cost −0.0088, which is below 0.0160. λ=0.05 was −0.0271 and λ=0.2 was −0.0070. |
| λ_cons (KL consistency) | 0.1 | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v13 grid: λ=0.2 +0.0008, λ=0.05 −0.0232. Removing L_cons cost −0.0165. |
| Ordinal-severity loss | Not used (rejected) | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v14 best λ_ord>0 0.8859 vs none 0.9048 (−0.0189); adjacent-class errors rose 94 → 113 |
| Classification loss | Cross-entropy weighted by inverse **training-set** class frequency (control) | **UNDER TEST IN v20:** A = weighted focal (γ=2), B = weighted ordinal label smoothing (τ=1), each *replacing* the CE term |
| Seeding | Python `random`, numpy, torch CPU + all CUDA devices, cuDNN deterministic; reset per seed at run start and right before model construction. The mission split has its own fixed seed (42), so every seed sees the same train/val/test windows. | CORRECTNESS FIX (reproducibility) |

## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [1]:
import copy
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table above)
# ==============================================================================
# --- Class-4 loss reshaping: 2 candidates x 5 seeds (the CE control is REUSED from v18, not retrained) ---
SEEDS = [42, 123, 2024, 7, 99]           # same 5 seeds as every prior multi-seed experiment
CANDIDATES = ["focal", "ordinal"]        # A then B; each over all SEEDS -> 10 new runs
RUN_ORDER = [(cand, seed) for cand in CANDIDATES for seed in SEEDS]
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 0.0                          # v20: HARD-FIXED at exactly 0.0 (beta question parked)
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params
LAG_BIAS_PARAMS = 97                     # the LayerNorm added in v18 has no parameters -> still 97
LAG_BIAS_LN_EPS = 1e-5                   # v18 LayerNorm over each window's (T_m, T_n) lag-bias grid (kept; inert at beta=0)
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL); v20 swaps L_cls's formula only ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07
FOCAL_GAMMA = 2.0                        # candidate A: alpha_y * (1 - p_t)^gamma * (-log p_t); fixed, not swept
ORDINAL_TAU = 1.0                        # candidate B: q(k|y) ~ exp(-|k - y| / tau); fixed, not swept
TARGET_CLASS = 4                         # the class whose recall is the hypothesis under test

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2
FOCAL_DIAGNOSTIC_EPOCH = EPOCHS // 2          # candidate A: p_t / (1-p_t)^gamma printed on real batch examples here

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v20_checkpoints"
PROBS_DIR = WORK_DIR / "v20_test_probs"
RESULTS_CSV = WORK_DIR / "class4_loss_reshaping_results.csv"
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- CONTROL: v18's beta=0.0 runs, REUSED (not retrained). Values copied from Artifacts/Version 18 CSV. ---
# v18 at beta=0 == this notebook's architecture with the class-weighted CE loss. init_sha1 / rng_sha1 are used to
# verify that every new run starts from exactly the control run's initial weights and RNG stream.
V18_BETA0_CONTROL = {
    42: {"accuracy": 0.9088495575221239, "macro_f1": 0.9057330017972403, "balanced_accuracy": 0.9030033658152373, "macro_auroc": 0.9891867537737307,
          "init_sha1": "5850841e699f", "rng_sha1": "5641b401720d"},
    123: {"accuracy": 0.8876106194690265, "macro_f1": 0.8826825575079713, "balanced_accuracy": 0.8790467961967972, "macro_auroc": 0.9861866752615566,
          "init_sha1": "74be0883c62b", "rng_sha1": "0a4b3be44b18"},
    2024: {"accuracy": 0.8805309734513275, "macro_f1": 0.8763643000413779, "balanced_accuracy": 0.8715485268344848, "macro_auroc": 0.9866958753709998,
          "init_sha1": "92852955eef1", "rng_sha1": "619c0e0a6b4d"},
    7: {"accuracy": 0.9061946902654867, "macro_f1": 0.9019350016739314, "balanced_accuracy": 0.9008047135870789, "macro_auroc": 0.9897445660698647,
          "init_sha1": "7f5d0cd2e7cb", "rng_sha1": "9fdcf777d285"},
    99: {"accuracy": 0.8690265486725663, "macro_f1": 0.8638026478536822, "balanced_accuracy": 0.8598248510633313, "macro_auroc": 0.9829841151452052,
          "init_sha1": "e40c78169f57", "rng_sha1": "81dcd4638a88"},
}
# v18's CSV has no confusion matrices -> the control's Class 4 recall / adjacent errors are rebuilt from v18's saved
# test probabilities. Attach v18's notebook output as a Kaggle input; these patterns are searched in order.
V18_PROBS_GLOBS = ["/kaggle/input/**/v18_test_probs/beta0.0_seed{seed}_test_probs.npy",
                   "/kaggle/working/v18_test_probs/beta0.0_seed{seed}_test_probs.npy"]
REGENERATE_CONTROL_IF_MISSING = True     # no v18 probabilities for a seed -> retrain that control run (must reproduce v18)
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1
PAIRED_ALPHA = 0.05
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v15)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"CLASS-4 LOSS RESHAPING: candidates {CANDIDATES} x seeds {SEEDS} = {len(RUN_ORDER)} new runs + the REUSED v18 "
      f"beta=0.0 control | results -> {RESULTS_CSV}")
print(f"A: weighted focal gamma={FOCAL_GAMMA} | B: weighted ordinal label smoothing tau={ORDINAL_TAU} | each REPLACES the "
      f"class-weighted CE term; lambda_align={LAMBDA_ALIGN}, lambda_cons={LAMBDA_CONS} unchanged | LACA beta hard-fixed at "
      f"{LACA_BETA}")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA beta={LACA_BETA} (fixed), {LACA_NUM_HEADS} heads, mlp g(Δτ) "
      f"{LAG_BIAS_PARAMS} params | Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | "
      f"loss L_cls + {LAMBDA_ALIGN} InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")

Device: cuda (Tesla T4)
CLASS-4 LOSS RESHAPING: candidates ['focal', 'ordinal'] x seeds [42, 123, 2024, 7, 99] = 10 new runs + the REUSED v18 beta=0.0 control | results -> /kaggle/working/class4_loss_reshaping_results.csv
A: weighted focal gamma=2.0 | B: weighted ordinal label smoothing tau=1.0 | each REPLACES the class-weighted CE term; lambda_align=0.1, lambda_cons=0.1 unchanged | LACA beta hard-fixed at 0.0
Fixed configuration: audio strongest-64, Ta=64, sqrt, no window, no normalization | sensor raw 6 ch, Ts=128 | LACA beta=0.0 (fixed), 4 heads, mlp g(Δτ) 97 params | Mamba 2 x (d_state 16, d_conv 2) | loss L_cls + 0.1 InfoNCE + 0.1 KL | 20 epochs, batch 32, AdamW 0.0001


In [2]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

Installing mamba_ssm + causal-conv1d (python=cp312, torch=2.10, cuda=12.8, cxx11abi=True)
    pip install ['https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl', '--no-deps'] failed (exit 1): ERROR: causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl is not a supported wheel on this platform.

  no complete prebuilt-wheel match — trying a time-capped source build
mamba_ssm install: source build
✅ FUSION BACKBONE: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 3. Data loading & preprocessing
Mission-level split, strongest-64 bin selection from training audio only, 2.56 s windows with real timestamps, and sensor standardization with training-split statistics. Leakage and timestamp checks are included.

In [3]:
# ==============================================================================
# DATA — TII UAV Realistic Fault Dataset: parse missions -> mission-level split
# -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows with
# real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
split_windows = {}
for split in ("train", "val", "test"):
    split_windows[split] = []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                split_windows[split].extend(slice_windows(parsed_missions[m], label, SELECTED_BINS))
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps reach HTT/LACA ✅")
del _tau_a, _tau_s

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git ...


Cloning into '/kaggle/working/tii_uav_dataset'...
Updating files: 100% (217/217), done.


Class 0 (19 flights) -> train 13, val 2, test 4
Class 1 (20 flights) -> train 14, val 3, test 3
Class 2 (20 flights) -> train 14, val 3, test 3
Class 3 (20 flights) -> train 14, val 3, test 3
Class 4 (20 flights) -> train 14, val 3, test 3


parse train:   0%|          | 0/69 [00:00<?, ?it/s]

parse val:   0%|          | 0/14 [00:00<?, ?it/s]

parse test:   0%|          | 0/16 [00:00<?, ?it/s]

[LEAKAGE CHECK] strongest-64 bins computed from 69 TRAINING missions only (0 of 30 val/test missions) -> frozen for all splits: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
train: 4934 windows | audio 64x64, sensor 128x6
val  : 954 windows | audio 64x64, sensor 128x6
test : 1130 windows | audio 64x64, sensor 128x6
[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | train mean [0.05299999937415123, -0.0430000014603138, -9.78600025177002, 0.004000000189989805, 0.003000000026077032, -0.0] vs val's own mean [0.029999999329447746, -0.061000000685453415, -9.748000144958496, 0.004999999888241291, 0.003000000026077032, -0.0] (not used)
[TIMESTAMP CHECK] 4934 training windows: 4934 distinct 

## 4. Model definition — β hard-fixed at 0.0
This is the v18 architecture, built with `beta=0.0`. The β warm-up schedule and `set_beta` are removed, so β stays exactly 0.0 for the whole run. The v18 LayerNorm and zero-init of g are unchanged, but at β = 0 they cannot affect the output.

In [4]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention; v18 lag bias beta * LN_grid(g(Δτ)) with beta
#    HARD-FIXED at 0.0 in v20 -> content-only attention; 4 heads)
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding + time embedding(tau)); modality 0 = audio, 1 = sensor."""

    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = the value in effect (warm-up schedule)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.beta_target = float(beta)   # v20: always 0.0, never changed during training

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model, beta=self.beta_target)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model, beta=LACA_BETA):
    """Finalized architecture, read from the constructed modules (beta = the value this model must use)."""
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != requested {beta}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe = AeroSyncMamba(beta=LACA_BETA)
print(f"Model: {sum(p.numel() for p in _probe.parameters()):,} params | fusion blocks: "
      f"{type(_probe.mamba_blocks[0]).__name__} x {len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} params) | "
      f"LACA {LACA_NUM_HEADS} heads, beta HARD-FIXED at {_probe.laca.laca_as.beta} / {_probe.laca.laca_sa.beta}, g(Δτ) "
      f"{LAG_BIAS_PARAMS} params | backbone: {FUSION_BACKBONE}")
del _probe


# --- LayerNorm placement: trace real tensor shapes through one LACA direction on 2 training windows ---
_ta = torch.stack([train_ds[i]["tau_a"] for i in range(2)])
_ts = torch.stack([train_ds[i]["tau_s"] for i in range(2)])
_probe = AeroSyncMamba(beta=LACA_BETA)
with torch.no_grad():
    for _name, _laca, _tm, _tn in (("laca_as", _probe.laca.laca_as, _ta, _ts), ("laca_sa", _probe.laca.laca_sa, _ts, _ta)):
        _dt = (_tm.unsqueeze(2) - _tn.unsqueeze(1)).unsqueeze(-1)
        _g = _laca.lag_bias(_dt).squeeze(-1)
        _out = _laca.lag_bias_term(_tm, _tn)
        print(f"[LAYERNORM] {_name}: Δτ {tuple(_dt.shape)} -> g(Δτ) {tuple(_g.shape)} = (B, T_m, T_n) -> "
              f"LN over dims {tuple(_g.shape[-2:])} ({_g.shape[-2] * _g.shape[-1]:,} values per window, per window) -> "
              f"{tuple(_out.shape)} -> x beta -> added to scores (B, {LACA_NUM_HEADS} heads, T_m, T_n)")
        assert float(_g.abs().max()) == 0.0 and float(_out.abs().max()) == 0.0, "zero-init g should give an all-zero term"
del _probe, _ta, _ts
print("[LAYERNORM] placement: PairwiseLACA.forward -> scores + beta * LN_grid(g(Δτ)); the LayerNorm is applied to g's "
      "scalar output BEFORE beta.")
print("[LAYERNORM] why this shape: g outputs one scalar per (i, j) pair, so there is no feature dim > 1 (LN over a "
      "size-1 dim would output 0 for every input). Normalizing over each window's full (T_m, T_n) grid covers every "
      "bias value in that window's attention matrix. It is per window, never across the batch, so a window's bias "
      "does not depend on its batch-mates and is the same in train and eval. No affine: beta is the only scale. The "
      "subtracted mean is one constant per window, which softmax ignores; only the std normalization affects "
      "attention.")

Model: 1,688,788 params | fusion blocks: MambaBlock x 2 (874,496 params) | LACA 4 heads, beta HARD-FIXED at 0.0 / 0.0, g(Δτ) 97 params | backbone: mamba_ssm.Mamba (real selective-scan SSM, CUDA)
[LAYERNORM] laca_as: Δτ (2, 64, 128, 1) -> g(Δτ) (2, 64, 128) = (B, T_m, T_n) -> LN over dims (64, 128) (8,192 values per window, per window) -> (2, 1, 64, 128) -> x beta -> added to scores (B, 4 heads, T_m, T_n)
[LAYERNORM] laca_sa: Δτ (2, 128, 64, 1) -> g(Δτ) (2, 128, 64) = (B, T_m, T_n) -> LN over dims (128, 64) (8,192 values per window, per window) -> (2, 1, 128, 64) -> x beta -> added to scores (B, 4 heads, T_m, T_n)
[LAYERNORM] placement: PairwiseLACA.forward -> scores + beta * LN_grid(g(Δτ)); the LayerNorm is applied to g's scalar output BEFORE beta.
[LAYERNORM] why this shape: g outputs one scalar per (i, j) pair, so there is no feature dim > 1 (LN over a size-1 dim would output 0 for every input). Normalizing over each window's full (T_m, T_n) grid covers every bias value in that windo

## 5. Losses, metrics and training function
**Losses.** `AeroSyncLoss(class_weights, cls_loss)` supports three versions of L_cls:
- `"ce"`: the control's exact `nn.CrossEntropyLoss(weight=α)`, used only if the control has to be regenerated;
- `"focal"`: candidate A;
- `"ordinal"`: candidate B.

InfoNCE and the KL consistency term are unchanged.

**Checks on real data before any training:**
- α_y is printed and compared with the training class counts;
- q(k | y) is printed for every class, and y = 3 is checked to give its neighbors more weight than distant classes;
- focal with γ = 0, and ordinal with one-hot targets, must both equal the control's weighted CE.

**Metrics.** Every run's full 5×5 confusion matrix, per-class recall, Class 4 recall, and adjacent vs distant errors (the same split as v14).

**Candidate A diagnostic.** At epoch 10, p_t and (1 − p_t)^γ are printed for real examples from the first training batch.

In [5]:
# ==============================================================================
# LOSSES (control CE / A: weighted focal / B: weighted ordinal label smoothing),
# metrics (4 standard + 5x5 confusion matrix + per-class recall + adjacent vs
# distant errors), loss sanity checks on real data, and train_one_run(seed,
# cls_loss): one full model end-to-end at beta = 0.0, evaluated on the test set.
# ==============================================================================
from sklearn.metrics import confusion_matrix

CLS_LOSS_LABELS = {"ce": "control: class-weighted CE",
                   "focal": f"A: weighted focal (gamma={FOCAL_GAMMA})",
                   "ordinal": f"B: weighted ordinal label smoothing (tau={ORDINAL_TAU})"}


def ordinal_soft_targets(n_classes=N_CLASSES, tau=ORDINAL_TAU):
    """Row y = q(. | y): q(k | y) = exp(-|k - y| / tau), normalized over k."""
    k = torch.arange(n_classes, dtype=torch.float32)
    q = torch.exp(-(k[None, :] - k[:, None]).abs() / tau)
    return q / q.sum(dim=1, keepdim=True)


class AeroSyncLoss(nn.Module):
    """L = L_cls + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency (unchanged structure).

    L_cls (alpha = per-class training weights, the SAME tensor for every variant):
      "ce"      : nn.CrossEntropyLoss(weight=alpha)                        — the control (v16-v18), unchanged
      "focal"   : sum_i alpha_yi (1 - p_t)^gamma (-log p_t)   / sum_i alpha_yi
      "ordinal" : sum_i alpha_yi (-sum_k q(k|yi) log p(k))    / sum_i alpha_yi
    Dividing by sum_i alpha_yi is exactly nn.CrossEntropyLoss(weight=alpha)'s 'mean' reduction, so focal with
    gamma = 0 and ordinal with one-hot targets both equal the control's L_cls (checked on real data below)."""

    def __init__(self, class_weights, cls_loss="ce", gamma=FOCAL_GAMMA, soft_targets=None):
        super().__init__()
        assert cls_loss in CLS_LOSS_LABELS, cls_loss
        self.cls_loss_name = cls_loss
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.class_weights = class_weights
        self.gamma = gamma
        self.soft_targets = (ordinal_soft_targets() if soft_targets is None else soft_targets).to(class_weights.device)
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def cls_loss(self, logits, targets):
        if self.cls_loss_name == "ce":
            return self.ce(logits, targets)
        logp = F.log_softmax(logits, dim=-1)
        alpha = self.class_weights[targets]
        if self.cls_loss_name == "focal":
            logp_t = logp.gather(1, targets.unsqueeze(1)).squeeze(1)
            per_example = -((1.0 - logp_t.exp()) ** self.gamma) * logp_t
        else:   # "ordinal"
            per_example = -(self.soft_targets[targets] * logp).sum(dim=-1)
        return (alpha * per_example).sum() / alpha.sum()

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.cls_loss(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        with torch.no_grad():   # scale reference: the control's weighted CE on the same logits (not in the graph)
            ce_ref = loss_cls.item() if self.cls_loss_name == "ce" else self.ce(logits, targets).item()
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item(),
                       "ce_ref": ce_ref}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
CLASS_WEIGHTS = train_class_weights(split_windows["train"]).to(device)
TEST_LABELS = np.array([w["label"] for w in split_windows["test"]], dtype=np.int64)   # test-window order, fixed


# --- alpha_y: the existing training-count weights, reused unchanged by every L_cls variant ---
_train_counts = np.bincount([w["label"] for w in split_windows["train"]], minlength=N_CLASSES)
_alpha_recomputed = _train_counts.sum() / (N_CLASSES * _train_counts)
assert tuple(_train_counts.tolist()) == REFERENCE_DATA["sensor_fingerprint"]["train"][3], "training class counts changed"
assert np.allclose(CLASS_WEIGHTS.cpu().numpy(), _alpha_recomputed, rtol=0, atol=1e-6)
print("[ALPHA] per-class weights alpha_y = N_train / (5 * n_y) — the existing train_class_weights(), unchanged, "
      "used by control, A and B:")
for c in range(N_CLASSES):
    print(f"    class {c}: n_train = {_train_counts[c]:>5} | alpha = {float(CLASS_WEIGHTS[c]):.6f}")
print(f"[ALPHA] ✅ matches the training counts {tuple(_train_counts.tolist())} (same as v16-v18). Note: class "
      f"{int(_train_counts.argmin())} is the smallest training class; alpha_{TARGET_CLASS} = "
      f"{float(CLASS_WEIGHTS[TARGET_CLASS]):.4f}, so class weighting gives Class {TARGET_CLASS} essentially no extra "
      f"weight — any Class {TARGET_CLASS} change must come from the reshaping.")

# --- candidate B: the soft target distributions ---
SOFT_TARGETS = ordinal_soft_targets()
print(f"\n[ORDINAL q(k|y)] tau = {ORDINAL_TAU}: q(k|y) = exp(-|k-y|/tau) / sum_k exp(-|k-y|/tau)")
print("            " + "".join(f"   k={k}  " for k in range(N_CLASSES)))
for y in range(N_CLASSES):
    print(f"    y = {y}  " + "".join(f"{float(SOFT_TARGETS[y, k]):>8.4f} " for k in range(N_CLASSES))
          + ("   <- example: middle class" if y == 3 else ""))
_q3 = SOFT_TARGETS[3]
assert _q3.argmax() == 3 and _q3[2] > _q3[1] > _q3[0] and _q3[4] > _q3[1] and torch.isclose(_q3.sum(), torch.tensor(1.0))
print(f"[ORDINAL q(k|3)] true class {float(_q3[3]):.4f} | neighbours 2 and 4: {float(_q3[2]):.4f} each | distance 2 "
      f"(k=1): {float(_q3[1]):.4f} | distance 3 (k=0): {float(_q3[0]):.4f} -> neighbours get {float(_q3[2] / _q3[1]):.2f}x "
      f"the mass of distance-2 classes ✅. The loss is minimized at p = q, whose argmax is still y.")

# --- both candidates reduce EXACTLY to the control's weighted CE in their limits (real labels, fixed logits) ---
_idx = np.linspace(0, len(train_ds) - 1, 64).astype(int)   # spread over the training set -> every class present
_y = torch.tensor([int(train_ds[i]["label"]) for i in _idx], device=device)
_logits = torch.randn(len(_idx), N_CLASSES, generator=torch.Generator().manual_seed(0)).to(device) * 2.0   # private RNG
_ce = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)(_logits, _y)
_focal0 = AeroSyncLoss(CLASS_WEIGHTS, "focal", gamma=0.0).cls_loss(_logits, _y)
_ord_onehot = AeroSyncLoss(CLASS_WEIGHTS, "ordinal", soft_targets=torch.eye(N_CLASSES)).cls_loss(_logits, _y)
_focal2 = AeroSyncLoss(CLASS_WEIGHTS, "focal").cls_loss(_logits, _y)
_ord1 = AeroSyncLoss(CLASS_WEIGHTS, "ordinal").cls_loss(_logits, _y)
assert torch.allclose(_focal0, _ce, atol=1e-6) and torch.allclose(_ord_onehot, _ce, atol=1e-6), (_ce, _focal0, _ord_onehot)
print(f"\n[LOSS CHECK] {len(_idx)} real training labels (classes {sorted(set(_y.tolist()))}), fixed logits: weighted CE "
      f"{float(_ce):.6f} | focal gamma=0 {float(_focal0):.6f} | ordinal one-hot {float(_ord_onehot):.6f} -> both reduce "
      f"to the control's L_cls ✅ | focal gamma=2 {float(_focal2):.6f}, ordinal tau=1 {float(_ord1):.6f} (min possible "
      f"= entropy of q, not 0)")
del _idx, _y, _logits, _ce, _focal0, _ord_onehot, _focal2, _ord1


def severity_error_breakdown(cm):
    """From a 5x5 confusion matrix (rows = true, cols = predicted, classes = severity 0..4): misclassifications
    with |pred - true| == 1 (ADJACENT severities) vs |pred - true| >= 2 (DISTANT), plus each adjacent pair (v14)."""
    cm = np.asarray(cm)
    n = cm.shape[0]
    dist = np.abs(np.arange(n)[:, None] - np.arange(n)[None, :])
    out = {
        "adjacent_errors": int(cm[dist == 1].sum()),
        "distant_errors": int(cm[dist >= 2].sum()),
        "total_errors": int(cm[dist >= 1].sum()),
    }
    for k in range(n - 1):   # both directions, e.g. err_2v3 = true 2 -> pred 3 plus true 3 -> pred 2
        out[f"err_{k}v{k + 1}"] = int(cm[k, k + 1] + cm[k + 1, k])
    return out


def evaluate_probs(y_true, y_prob):
    """4 standard metrics + full 5x5 confusion matrix + per-class recall + adjacent / distant errors."""
    cm = confusion_matrix(y_true, y_prob.argmax(axis=1), labels=list(range(N_CLASSES)))
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {**compute_metrics(y_true, y_prob), "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class4_recall": float(recall[TARGET_CLASS]), **severity_error_breakdown(cm)}


def print_confusion(ev, tag):
    cm = ev["confusion_matrix"]
    print(f"[CONFUSION {tag}] rows = TRUE severity, cols = PREDICTED severity (test set, {int(cm.sum())} windows)")
    print("            " + "".join(f"pred {j:>2} " for j in range(N_CLASSES)) + "|  recall")
    for i in range(N_CLASSES):
        print(f"   true {i:>2} " + "".join(f"{int(cm[i, j]):>8}" for j in range(N_CLASSES)) + f"  |  {ev[f'recall_c{i}']:.4f}"
              + ("   <- Class 4" if i == TARGET_CLASS else ""))
    print(f"[SEVERITY ERRORS {tag}] Class 4 recall {ev['class4_recall']:.4f} | ADJACENT (|pred-true|==1): "
          f"{ev['adjacent_errors']} | DISTANT (|pred-true|>=2): {ev['distant_errors']} | total {ev['total_errors']} | "
          + " | ".join(f"{k[4]}<->{k[6]}: {ev[k]}" for k in ev if k.startswith("err_")))


# Fixed probe batch for the lag-contribution check: timestamps of the first BATCH_SIZE training windows, read by
# index (no DataLoader, no RNG). The lag-bias term depends only on the timestamps.
LAG_PROBE_TAUS = tuple(torch.stack([train_ds[i][k] for i in range(min(BATCH_SIZE, len(train_ds)))])
                       for k in ("tau_a", "tau_s"))


def lag_bias_contribution(model):
    """Per LACA direction: max|beta * LN_grid(g(Δτ))|, the exact term PairwiseLACA adds to the attention scores
    (computed by the same lag_bias_term method, with the beta currently in effect), plus the raw std of g before
    the LayerNorm. g has no dropout or BatchNorm: this uses no random numbers and changes no state."""
    ta, ts = (t.to(device) for t in LAG_PROBE_TAUS)
    out = {}
    with torch.no_grad():
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts), ("laca_sa", model.laca.laca_sa, ts, ta)):
            term = laca.lag_bias_term(tau_m, tau_n)
            contrib = laca.beta * term
            g_raw = laca.lag_bias((tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)).squeeze(-1)
            out[name] = {"beta": laca.beta, "contrib_absmax": float(contrib.abs().max()),
                         "contrib_finite": bool(torch.isfinite(contrib).all()), "g_raw_std": float(g_raw.std())}
    return out


def g_final_layer_absmax(model):
    """max|weight| and max|bias| of g's final Linear(32, 1), per LACA direction."""
    return {name: (float(getattr(model.laca, name).lag_bias[-1].weight.detach().abs().max()),
                   float(getattr(model.laca, name).lag_bias[-1].bias.detach().abs().max()))
            for name in ("laca_as", "laca_sa")}


def focal_diagnostic(logits, targets, tag, epoch, n_show=8):
    """Candidate A, on REAL training examples: p_t and the (1 - p_t)^gamma down-weighting factor. Shows the 4
    lowest-p_t and 4 highest-p_t examples of the batch, and batch-level summaries."""
    with torch.no_grad():
        p = F.softmax(logits.float(), dim=-1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        factor = (1.0 - p_t) ** FOCAL_GAMMA
        ce_i = -torch.log(p_t.clamp_min(1e-12))
        alpha = CLASS_WEIGHTS[targets]
    order = torch.argsort(p_t).tolist()
    show = order[: n_show // 2] + order[-(n_show // 2):]
    print(f"{tag} [FOCAL DIAGNOSTIC, epoch {epoch}, first training batch ({len(targets)} real examples)]")
    print(f"      {'ex':>3} {'true y':>6} {'p_t':>8} {'(1-p_t)^g':>10} {'alpha_y':>8} {'CE_i':>8} {'focal_i':>9}")
    for i in show:
        print(f"      {i:>3} {int(targets[i]):>6} {float(p_t[i]):>8.4f} {float(factor[i]):>10.5f} {float(alpha[i]):>8.4f} "
              f"{float(ce_i[i]):>8.4f} {float(factor[i] * ce_i[i]):>9.5f}")
    print(f"{tag} [FOCAL DIAGNOSTIC] batch: mean p_t {float(p_t.mean()):.4f} | mean (1-p_t)^gamma {float(factor.mean()):.4f} | "
          f"{int((factor < 0.01).sum())}/{len(factor)} examples down-weighted >100x | easiest example weighted "
          f"{float(factor.min()):.2e}, hardest {float(factor.max()):.3f} -> confident examples contribute ~nothing, hard "
          f"ones keep ~full CE weight ✅")
    return float(factor.mean())


def train_one_run(seed: int, cls_loss: str):
    """Train one model at beta = 0.0 with `seed` applied to every RNG and L_cls = `cls_loss`, evaluate on the
    test set. Returns (trained model on CPU, result dict incl. test probabilities and confusion matrix)."""
    t0 = time.time()
    tag = f"[{cls_loss} seed={seed}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(beta=LACA_BETA).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model, beta=LACA_BETA)
    assert (model.laca.laca_as.beta, model.laca.laca_sa.beta) == (0.0, 0.0)
    ctrl = V18_BETA0_CONTROL[seed]
    paired_ok = init_sha1[:12] == ctrl["init_sha1"] and rng_sha1[:12] == ctrl["rng_sha1"]
    print(f"{tag} L_cls = {CLS_LOSS_LABELS[cls_loss]} | beta = 0.0 (both directions) | fusion "
          f"{type(model.mamba_blocks[0]).__name__} ({mamba_params:,} params)")
    print(f"{tag} [PAIRING] initial weights {init_sha1[:12]} vs control {ctrl['init_sha1']} | torch RNG after construction "
          f"{rng_sha1[:12]} vs control {ctrl['rng_sha1']} -> "
          + ("✅ identical start and RNG stream to the v18 beta=0 control run — only L_cls differs" if paired_ok else
             "🚨 MISMATCH — this run does not start where the control started; the pairing is broken"))

    criterion = AeroSyncLoss(CLASS_WEIGHTS, cls_loss)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    focal_mean_factor = float("nan")
    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        for step, batch in enumerate(tqdm(train_loader, desc=f"{cls_loss} seed {seed} epoch {epoch:02d}/{EPOCHS}", leave=False)):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            if cls_loss == "focal" and epoch == FOCAL_DIAGNOSTIC_EPOCH and step == 0:
                focal_mean_factor = focal_diagnostic(logits.detach(), y, tag, epoch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        lag = lag_bias_contribution(model)
        assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"beta=0 leak at epoch {epoch}"

        val_probs, val_labels = predict_probs(model, val_loader)
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        n = len(train_loader)
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | train loss {running['loss'] / n:.4f} (L_cls {running['loss_cls'] / n:.4f} "
              f"[weighted-CE on same logits {running['ce_ref'] / n:.4f}], align {running['loss_align'] / n:.4f} x "
              f"{LAMBDA_ALIGN}, cons {running['loss_cons'] / n:.4f} x {LAMBDA_CONS}) | lag term 0.0 ✅ | val macro-F1 "
              f"{val_f1:.4f}{'  * best' if improved else ''}")

    model.load_state_dict(best_state)
    test_probs, test_labels = predict_probs(model, test_loader)
    ev = evaluate_probs(test_labels, test_probs)
    run_name = f"{cls_loss}_seed{seed}"
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "cls_loss": cls_loss,
        "seed": seed,
        **ev,
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "n_test_windows": len(test_labels),
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "paired_with_control": paired_ok,
        "focal_mean_downweight_diag": focal_mean_factor,
        "final_train_loss_cls": running["loss_cls"] / n,
        "final_train_ce_ref": running["ce_ref"] / n,
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model.cpu(), result

[ALPHA] per-class weights alpha_y = N_train / (5 * n_y) — the existing train_class_weights(), unchanged, used by control, A and B:
    class 0: n_train =   906 | alpha = 1.089183
    class 1: n_train =  1020 | alpha = 0.967451
    class 2: n_train =  1004 | alpha = 0.982869
    class 3: n_train =  1008 | alpha = 0.978968
    class 4: n_train =   996 | alpha = 0.990763
[ALPHA] ✅ matches the training counts (906, 1020, 1004, 1008, 996) (same as v16-v18). Note: class 0 is the smallest training class; alpha_4 = 0.9908, so class weighting gives Class 4 essentially no extra weight — any Class 4 change must come from the reshaping.

[ORDINAL q(k|y)] tau = 1.0: q(k|y) = exp(-|k-y|/tau) / sum_k exp(-|k-y|/tau)
               k=0     k=1     k=2     k=3     k=4  
    y = 0    0.6364   0.2341   0.0861   0.0317   0.0117 
    y = 1    0.1915   0.5206   0.1915   0.0705   0.0259 
    y = 2    0.0675   0.1834   0.4984   0.1834   0.0675 
    y = 3    0.0259   0.0705   0.1915   0.5206   0.1915    <- exa

## 6. Control reuse, then 10 new runs
**1. Control (reused).** For each seed, the driver loads v18's saved β = 0.0 test probabilities and rebuilds the confusion matrix. The 4 metrics computed from those probabilities must equal the v18 values embedded in Section 2. No training happens for these. If a seed's probabilities aren't attached and `REGENERATE_CONTROL_IF_MISSING` is on, that control run is retrained. It must reproduce v18's metrics, and the output says so loudly.

**2. New runs.** Candidate A on all 5 seeds, then candidate B on all 5 seeds. After each run the driver:
- prints the full confusion matrix;
- appends a row to `/kaggle/working/class4_loss_reshaping_results.csv`, with the confusion matrix stored as JSON;
- frees the GPU and prints progress.

In [6]:
# ==============================================================================
# DRIVER — (1) CONTROL: reuse v18's beta=0.0 runs (rebuild their confusion
# matrices from v18's saved test probabilities; verify they equal the v18
# values); regenerate only if missing. (2) Candidate A x 5 seeds, then B x 5.
# Every run: full confusion matrix printed, CSV row, GPU freed, progress.
# ==============================================================================
import glob

import pandas as pd

RESULTS_CSV_FIELDS = (["run", "cls_loss", "seed", "source", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc",
                       "class4_recall"] + [f"recall_c{c}" for c in range(N_CLASSES)]
                      + ["adjacent_errors", "distant_errors", "total_errors"] + [f"err_{k}v{k + 1}" for k in range(N_CLASSES - 1)]
                      + ["confusion_matrix", "best_val_macro_f1", "best_epoch", "n_test_windows", "init_sha1", "rng_sha1",
                         "paired_with_control", "focal_mean_downweight_diag", "final_train_loss_cls", "final_train_ce_ref",
                         "train_seconds", "finished_at"])
METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]

if RESULTS_CSV.exists():
    RESULTS_CSV.rename(RESULTS_CSV.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))
_run_counter = [0]


def log_row(result, source):
    _run_counter[0] += 1
    write_header = not RESULTS_CSV.exists()
    row = {k: result[k] for k in RESULTS_CSV_FIELDS if k in result}
    row.update(run=_run_counter[0], source=source, confusion_matrix=json.dumps(np.asarray(result["confusion_matrix"]).tolist()),
               finished_at=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---------------------------------------------------------------- (1) CONTROL
print("=" * 100)
print("CONTROL GROUP — v18's beta=0.0 runs, REUSED (not retrained). v18 at beta=0 IS this notebook's architecture with")
print("the class-weighted CE loss. Their confusion matrices are rebuilt from v18's saved test probabilities.")
print("=" * 100)
control, CONTROL_SOURCE, CONTROL_MATCH = {}, {}, {}
for seed in SEEDS:
    ref = V18_BETA0_CONTROL[seed]
    hits = [h for pattern in V18_PROBS_GLOBS for h in sorted(glob.glob(pattern.format(seed=seed), recursive=True))]
    ev = None
    for h in hits:
        probs = np.load(h)
        labels_path = Path(h.replace("_test_probs.npy", "_test_labels.npy"))
        labels = np.load(labels_path) if labels_path.exists() else TEST_LABELS
        if probs.shape != (len(TEST_LABELS), N_CLASSES) or not np.array_equal(labels, TEST_LABELS):
            print(f"[CONTROL seed={seed}] {h}: wrong shape / test order — ignored")
            continue
        cand = evaluate_probs(TEST_LABELS, probs)
        worst = max(abs(cand[m] - ref[m]) for m in METRICS)
        if worst <= SAME_SEED_TOLERANCE:
            ev, CONTROL_SOURCE[seed] = cand, f"reused v18 probabilities ({h})"
            CONTROL_MATCH[seed] = worst
            break
        print(f"[CONTROL seed={seed}] {h}: metrics differ from v18 by {worst:.4f} — not the v18 control, ignored")
    if ev is None:
        if not REGENERATE_CONTROL_IF_MISSING:
            print(f"🚨 [CONTROL seed={seed}] v18 probabilities NOT FOUND and regeneration is off -> Class 4 recall / "
                  f"adjacent errors unavailable for this seed's control (macro-F1 from the embedded v18 values only).")
            control[seed] = {**{m: ref[m] for m in METRICS}, "class4_recall": float("nan"), "adjacent_errors": float("nan")}
            CONTROL_SOURCE[seed] = "embedded v18 metrics only (no confusion matrix)"
            continue
        print("⚠️ " * 33)
        print(f"⚠️  [CONTROL seed={seed}] v18's saved test probabilities were NOT FOUND (attach v18's notebook output to "
              f"reuse them). REGENERATING this control run — it must reproduce v18 exactly, so it is the SAME control.")
        print("⚠️ " * 33)
        model, res = train_one_run(seed, "ce")
        del model
        free_gpu()
        ev = res
        CONTROL_MATCH[seed] = max(abs(res[m] - ref[m]) for m in METRICS)
        CONTROL_SOURCE[seed] = "regenerated (v18 probabilities not attached)"
        if CONTROL_MATCH[seed] > SAME_SEED_TOLERANCE:
            print("🚨" * 45 + f"\n🚨  regenerated control seed {seed} does NOT reproduce v18 (max |delta| "
                  f"{CONTROL_MATCH[seed]:.4f}). The regenerated run is used for pairing (same environment as the "
                  f"candidates), but it is not v18's control.\n" + "🚨" * 45)
    control[seed] = ev
    print(f"[CONTROL seed={seed}] {CONTROL_SOURCE[seed]}")
    print(f"[CONTROL seed={seed}] vs v18: " + " | ".join(f"{m} {ev[m]:.4f} (v18 {ref[m]:.4f})" for m in METRICS)
          + f" -> max |delta| {CONTROL_MATCH[seed]:.1e} "
          + ("✅ EXACT match" if CONTROL_MATCH[seed] < 1e-9 else ("✅ match" if CONTROL_MATCH[seed] <= SAME_SEED_TOLERANCE else "❌")))
    print_confusion(ev, f"control seed={seed}")
    log_row({"cls_loss": "ce", "seed": seed, **ev, "n_test_windows": len(TEST_LABELS),
             "init_sha1": V18_BETA0_CONTROL[seed]["init_sha1"], "rng_sha1": V18_BETA0_CONTROL[seed]["rng_sha1"]},
            CONTROL_SOURCE[seed])

# ---------------------------------------------------------------- (2) 10 NEW RUNS
run_results = {}
t_driver = time.time()
for i, (cand, seed) in enumerate(RUN_ORDER, start=1):
    print("\n" + "=" * 100 + f"\nRUN {i}/{len(RUN_ORDER)} — {CLS_LOSS_LABELS[cand]}, seed={seed}\n" + "=" * 100)
    model, result = train_one_run(seed, cand)
    run_results[(cand, seed)] = result
    print(f"{'[' + cand + ' seed=' + str(seed) + ']'} TEST: accuracy {result['accuracy']:.4f} | macro-F1 {result['macro_f1']:.4f} | "
          f"balanced accuracy {result['balanced_accuracy']:.4f} | macro-AUROC {result['macro_auroc']:.4f} | best epoch "
          f"{result['best_epoch']} | {result['train_seconds']:.0f}s")
    print_confusion(result, f"{cand} seed={seed}")
    c = control[seed]
    print(f"[PAIR {cand} seed={seed}] vs control: macro-F1 {result['macro_f1'] - c['macro_f1']:+.4f} | Class 4 recall "
          f"{result['class4_recall'] - c['class4_recall']:+.4f} | adjacent errors "
          f"{result['adjacent_errors'] - c['adjacent_errors']:+.0f}")
    log_row(result, "new run")
    del model
    free_gpu()
    elapsed = time.time() - t_driver
    print(f"\n>>> {i}/{len(RUN_ORDER)} complete: {cand} seed={seed} finished | elapsed {elapsed / 60:.1f} min | "
          f"est. remaining {elapsed / i * (len(RUN_ORDER) - i) / 60:.1f} min")

print(f"\nAll {len(RUN_ORDER)} new runs finished in {(time.time() - t_driver) / 60:.1f} min. Results: {RESULTS_CSV}")

CONTROL GROUP — v18's beta=0.0 runs, REUSED (not retrained). v18 at beta=0 IS this notebook's architecture with
the class-weighted CE loss. Their confusion matrices are rebuilt from v18's saved test probabilities.
⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ 
⚠️  [CONTROL seed=42] v18's saved test probabilities were NOT FOUND (attach v18's notebook output to reuse them). REGENERATING this control run — it must reproduce v18 exactly, so it is the SAME control.
⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ ⚠️ 
[ce seed=42] L_cls = control: class-weighted CE | beta = 0.0 (both directions) | fusion MambaBlock (874,496 params)
[ce seed=42] [PAIRING] initial weights 5850841e699f vs control 5850841e699f | torch RNG after construction 5641b401720d vs control 5641b401720d -> ✅ identical start and RNG stream to the v18 beta=0 control run — only L_cls differs


ce seed 42 epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

[ce seed=42] epoch 01/20 | train loss 1.4174 (L_cls 1.0707 [weighted-CE on same logits 1.0707], align 3.1025 x 0.1, cons 0.3650 x 0.1) | lag term 0.0 ✅ | val macro-F1 0.8261  * best


ce seed 42 epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

[ce seed=42] epoch 02/20 | train loss 0.9259 (L_cls 0.5977 [weighted-CE on same logits 0.5977], align 2.6650 x 0.1, cons 0.6171 x 0.1) | lag term 0.0 ✅ | val macro-F1 0.6814


...

[ordinal seed=99] epoch 18/20 | train loss 1.2031 (L_cls 1.1814 [weighted-CE on same logits 0.6222], align 0.1527 x 0.1, cons 0.0642 x 0.1) | lag term 0.0 ✅ | val macro-F1 0.8903


ordinal seed 99 epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

[ordinal seed=99] epoch 19/20 | train loss 1.2021 (L_cls 1.1806 [weighted-CE on same logits 0.6186], align 0.1503 x 0.1, cons 0.0642 x 0.1) | lag term 0.0 ✅ | val macro-F1 0.9006


ordinal seed 99 epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

[ordinal seed=99] epoch 20/20 | train loss 1.2015 (L_cls 1.1805 [weighted-CE on same logits 0.6168], align 0.1460 x 0.1, cons 0.0647 x 0.1) | lag term 0.0 ✅ | val macro-F1 0.9089
[ordinal seed=99] TEST: accuracy 0.8894 | macro-F1 0.8851 | balanced accuracy 0.8812 | macro-AUROC 0.9809 | best epoch 10 | 223s
[CONFUSION ordinal seed=99] rows = TRUE severity, cols = PREDICTED severity (test set, 1130 windows)
            pred  0 pred  1 pred  2 pred  3 pred  4 |  recall
   true  0      283       1       2       0       0  |  0.9895
   true  1        7     192       6       0       0  |  0.9366
   true  2        0       3     180      27       0  |  0.8571
   true  3        0       0      19     201       8  |  0.8816
   true  4        1       1       2      48     149  |  0.7413   <- Class 4
[SEVERITY ERRORS ordinal seed=99] Class 4 recall 0.7413 | ADJACENT (|pred-true|==1): 119 | DISTANT (|pred-true|>=2): 6 | total 125 | 0<->1: 8 | 1<->2: 9 | 2<->3: 46 | 3<->4: 56
[PAIR ordinal seed=99] v

## 7. Paired statistics per candidate, and the summary
Each candidate is compared separately with the reused control, seed by seed. Three quantities: **macro-F1**, **Class 4 recall** (the hypothesis) and **adjacent errors** (the mechanism; lower is better). For each: the per-seed paired differences, the mean paired difference, the std of the paired differences, the win count, a **paired t-test (primary)** and a Wilcoxon signed-rank test (secondary).

**Why the t-test is primary.** With 5 pairs, the exact two-sided Wilcoxon p-value can never go below 2/2⁵ = 0.0625, so it can never reach 0.05. Class 4 recall moves in steps of 1/201 ≈ 0.005, so tied ranks are likely, which weakens Wilcoxon further. The t-test uses the size of each difference. It assumes the differences are roughly normal, which can't be checked with 5 pairs, so the Wilcoxon test is reported alongside it.

**Summary.** One table (control / A / B) with mean macro-F1, mean Class 4 recall and mean adjacent errors, then a plain verdict for each candidate. On Class 4 recall, a candidate **CLEARLY EXCEEDS THE PAIRED NOISE** only if its mean paired difference is positive **and** larger than the std of its paired differences. Recall for every class is also printed, so class 2 stays visible.

In [7]:
# ==============================================================================
# ANALYSIS — per-candidate paired statistics vs the reused control (macro-F1,
# Class 4 recall, adjacent errors), per-class recall, summary + verdicts.
# ==============================================================================
from scipy import stats

df = pd.read_csv(RESULTS_CSV)
ctrl_df = df[df["cls_loss"] == "ce"].set_index("seed")
QUANTITIES = [("macro_f1", "macro-F1", +1), ("class4_recall", "Class 4 recall", +1), ("adjacent_errors", "adjacent errors", -1)]


def paired_stats(new, base, better_sign=+1):
    d = np.asarray(new, dtype=float) - np.asarray(base, dtype=float)
    k = len(d)
    out = {"d": d, "n": k, "mean": float(d.mean()), "sd": float(d.std(ddof=1)) if k > 1 else float("nan"),
           "wins": int((better_sign * d > 0).sum()), "losses": int((better_sign * d < 0).sum()), "ties": int((d == 0).sum())}
    out["p_t"] = float(stats.ttest_rel(new, base).pvalue) if k > 1 and out["sd"] > 0 else float("nan")
    try:
        out["p_w"] = float(stats.wilcoxon(d, alternative="two-sided", method="exact").pvalue)
    except Exception:
        try:
            out["p_w"] = float(stats.wilcoxon(d, alternative="two-sided").pvalue)   # ties / zeros -> normal approx.
        except Exception:
            out["p_w"] = float("nan")
    return out


RESULTS = {}
for cand in CANDIDATES:
    cdf = df[df["cls_loss"] == cand].set_index("seed")
    seeds = [s for s in SEEDS if s in cdf.index and s in ctrl_df.index]
    print("\n" + "=" * 110)
    print(f"{CLS_LOSS_LABELS[cand].upper()} vs REUSED CONTROL — {len(seeds)} matched seeds {seeds}")
    print("=" * 110)
    RESULTS[cand] = {}
    for q, label, sign in QUANTITIES:
        ok = [s for s in seeds if not (pd.isna(ctrl_df.loc[s, q]) or pd.isna(cdf.loc[s, q]))]
        if len(ok) < 2:
            print(f"  {label}: fewer than 2 seeds with control data — not computable")
            continue
        base = ctrl_df.loc[ok, q].astype(float).to_numpy()
        new = cdf.loc[ok, q].astype(float).to_numpy()
        P = paired_stats(new, base, sign)
        RESULTS[cand][q] = {**P, "seeds": ok, "base_mean": base.mean(), "new_mean": new.mean()}
        fmt = (lambda v: f"{v:>9.0f}") if q == "adjacent_errors" else (lambda v: f"{v:>9.4f}")
        dfmt = (lambda v: f"{v:>+9.0f}") if q == "adjacent_errors" else (lambda v: f"{v:>+9.4f}")
        print(f"\n  {label}{' (lower is better)' if sign < 0 else ''}")
        print(f"    {'seed':>6} {'control':>9} {cand:>9} {'diff':>9}  better?")
        for s, b0, b1, x in zip(ok, base, new, P["d"]):
            print(f"    {s:>6} {fmt(b0)} {fmt(b1)} {dfmt(x)}  {'yes' if sign * x > 0 else ('tie' if x == 0 else 'no')}")
        print(f"    mean paired difference {P['mean']:+.4f} | std of paired differences {P['sd']:.4f} | "
              f"{cand} better on {P['wins']}/{P['n']} seeds (worse {P['losses']}, ties {P['ties']})")
        print(f"    paired t-test (PRIMARY) two-sided p = {P['p_t']:.4f} | Wilcoxon signed-rank p = {P['p_w']:.4f} "
              f"(smallest achievable with n={P['n']}: {2 / 2 ** P['n']:.4f})")

# --- per-class recall means (every class, so class 2 stays visible) ---
print("\n" + "=" * 110)
print("MEAN RECALL PER CLASS across seeds")
print("=" * 110)
print(f"  {'':<42}" + "".join(f"{'class ' + str(c):>10}" for c in range(N_CLASSES)))
for name, sub in [("control (reused v18 beta=0)", ctrl_df)] + [(CLS_LOSS_LABELS[c], df[df["cls_loss"] == c].set_index("seed"))
                                                               for c in CANDIDATES]:
    print(f"  {name:<42}" + "".join(f"{sub[f'recall_c{c}'].astype(float).mean():>10.4f}" for c in range(N_CLASSES)))

# --- summary table + verdicts ---
print("\n" + "#" * 110)
print("#  SUMMARY — means across seeds (control = REUSED v18 beta=0.0 runs)")
print("#" * 110)
print(f"#  {'':<48} {'macro-F1':>9} {'Class 4 recall':>15} {'adjacent errors':>16}")
print(f"#  {'control: class-weighted CE (reused)':<48} {ctrl_df['macro_f1'].astype(float).mean():>9.4f} "
      f"{ctrl_df['class4_recall'].astype(float).mean():>15.4f} {ctrl_df['adjacent_errors'].astype(float).mean():>16.1f}")
for cand in CANDIDATES:
    sub = df[df["cls_loss"] == cand]
    print(f"#  {CLS_LOSS_LABELS[cand]:<48} {sub['macro_f1'].astype(float).mean():>9.4f} "
          f"{sub['class4_recall'].astype(float).mean():>15.4f} {sub['adjacent_errors'].astype(float).mean():>16.1f}")
print("#")
VERDICTS = {}
for cand in CANDIDATES:
    r = RESULTS[cand]
    c4, f1, adj = r.get("class4_recall"), r.get("macro_f1"), r.get("adjacent_errors")
    if c4 is None:
        VERDICTS[cand] = "NOT COMPUTABLE (no control Class 4 recall)"
    elif c4["mean"] > 0 and c4["mean"] > c4["sd"]:
        VERDICTS[cand] = "CLEARLY EXCEEDS PAIRED NOISE"
    elif c4["mean"] < 0 and -c4["mean"] > c4["sd"]:
        VERDICTS[cand] = "CLEARLY WORSE (beyond paired noise)"
    else:
        VERDICTS[cand] = "WITHIN PAIRED NOISE"
    print(f"#  {CLS_LOSS_LABELS[cand]}:")
    if c4 is not None:
        print(f"#    Class 4 recall : {VERDICTS[cand]} — mean paired diff {c4['mean']:+.4f} vs std {c4['sd']:.4f} | better on "
              f"{c4['wins']}/{c4['n']} seeds | t-test p = {c4['p_t']:.4f}")
    if f1 is not None:
        f1_v = ("better beyond noise" if f1["mean"] > 0 and f1["mean"] > f1["sd"] else
                "worse beyond noise" if f1["mean"] < 0 and -f1["mean"] > f1["sd"] else "within noise")
        print(f"#    macro-F1       : {f1_v} — mean paired diff {f1['mean']:+.4f} vs std {f1['sd']:.4f} | better on "
              f"{f1['wins']}/{f1['n']} | p = {f1['p_t']:.4f}")
    if adj is not None:
        adj_v = ("FEWER beyond noise" if adj["mean"] < 0 and -adj["mean"] > adj["sd"] else
                 "MORE beyond noise" if adj["mean"] > 0 and adj["mean"] > adj["sd"] else "within noise")
        print(f"#    adjacent errors: {adj_v} — mean paired diff {adj['mean']:+.1f} vs std {adj['sd']:.1f} | fewer on "
              f"{adj['wins']}/{adj['n']} | p = {adj['p_t']:.4f}")
winners = [c for c in CANDIDATES if VERDICTS[c] == "CLEARLY EXCEEDS PAIRED NOISE"]
print("#")
if not winners:
    print("#  VERDICT: NEITHER candidate improves Class 4 recall by more than the paired noise. Neither the focal nor the "
          "ordinal reshaping of L_cls helps the target class on this evidence.")
elif len(winners) == 1:
    w = winners[0]
    print(f"#  VERDICT: {CLS_LOSS_LABELS[w]} is the ONLY candidate whose Class 4 recall gain clearly exceeds the paired "
          f"noise (+{RESULTS[w]['class4_recall']['mean']:.4f} vs std {RESULTS[w]['class4_recall']['sd']:.4f}); the "
          f"other does not.")
else:
    print("#  VERDICT: BOTH candidates improve Class 4 recall by more than the paired noise: "
          + "; ".join(f"{CLS_LOSS_LABELS[w]} +{RESULTS[w]['class4_recall']['mean']:.4f} (std "
                      f"{RESULTS[w]['class4_recall']['sd']:.4f})" for w in winners))
print("#  (One Class 4 test window = 1/201 = 0.0050 recall. The mean > std rule is the same bar the beta experiments used;")
print("#   with 5 seeds it corresponds to t > 2.24, i.e. p of about 0.09, which is weaker than p < 0.05.)")
print("#" * 110)


A: WEIGHTED FOCAL (GAMMA=2.0) vs REUSED CONTROL — 5 matched seeds [42, 123, 2024, 7, 99]

  macro-F1
      seed   control     focal      diff  better?
        42    0.9057    0.8912   -0.0145  no
       123    0.8827    0.8651   -0.0176  no
      2024    0.8764    0.8785   +0.0021  yes
         7    0.9019    0.8753   -0.0267  no
        99    0.8638    0.8908   +0.0270  yes
    mean paired difference -0.0059 | std of paired differences 0.0211 | focal better on 2/5 seeds (worse 3, ties 0)
    paired t-test (PRIMARY) two-sided p = 0.5638 | Wilcoxon signed-rank p = 0.8125 (smallest achievable with n=5: 0.0625)

  Class 4 recall
      seed   control     focal      diff  better?
        42    0.8507    0.8408   -0.0100  no
       123    0.7164    0.6567   -0.0597  no
      2024    0.7463    0.7612   +0.0149  yes
         7    0.8458    0.7910   -0.0547  no
        99    0.7413    0.7861   +0.0448  yes
    mean paired difference -0.0129 | std of paired differences 0.0449 | focal better on 

## 8. Final validation
A recap of every check: data and RNG diagnostics, α_y, control reuse (source and match with v18), pairing, β = 0 throughout, all 10 confusion matrices printed, and the verdicts.

In [8]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place
# ==============================================================================
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)
data_checks = {
    "strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
    **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
       REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")},
    "seed-42 torch RNG after construction == v16/v18": run_results[("focal", 42)]["rng_sha1"].startswith(REFERENCE_SEED42_RNG_PREFIX),
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in data_checks.items():
    print(f"  {name:<58}: {'match' if ok else 'MISMATCH'}")

pair_ok = {k: r["paired_with_control"] for k, r in run_results.items()}
n_reused = sum(v.startswith("reused") for v in CONTROL_SOURCE.values())
print("\n" + "=" * 100)
print(f"  alpha_y (training-count weights)            : " + ", ".join(f"{float(w):.4f}" for w in CLASS_WEIGHTS) + " ✅ unchanged")
print(f"  control: reused v18 runs (no retraining)    : {n_reused}/{len(SEEDS)} seeds"
      + ("" if n_reused == len(SEEDS) else f" | other seeds: {sorted(set(v for v in CONTROL_SOURCE.values() if not v.startswith('reused')))}"))
print(f"  control metrics vs known v18 values         : " + ", ".join(
      f"{s}: {'EXACT' if CONTROL_MATCH.get(s, 1) < 1e-9 else ('match' if CONTROL_MATCH.get(s, 1) <= SAME_SEED_TOLERANCE else 'MISMATCH/NA')}"
      for s in SEEDS) + " | macro-F1 " + ", ".join(f"{V18_BETA0_CONTROL[s]['macro_f1']:.4f}" for s in SEEDS))
print(f"  new runs start where the control started    : {'✅ all 10' if all(pair_ok.values()) else '🚨 ' + str({k: v for k, v in pair_ok.items() if not v})}")
print(f"  beta exactly 0.0, lag term 0 at every epoch : ✅ asserted in every run")
print(f"  confusion matrices printed                  : {len(run_results)} new runs + {len(control)} controls")
print(f"  focal down-weighting diagnostic (epoch {FOCAL_DIAGNOSTIC_EPOCH})  : mean (1-p_t)^gamma "
      + ", ".join(f"seed {s}: {run_results[('focal', s)]['focal_mean_downweight_diag']:.4f}" for s in SEEDS if ("focal", s) in run_results))
for cand in CANDIDATES:
    print(f"  VERDICT {cand:<8} on Class 4 recall          : {VERDICTS[cand]}")
print("=" * 100)
if not all(data_checks.values()):
    print("🚨 a data / RNG diagnostic is a MISMATCH — the data or model construction differs from v18; the reused control "
          "may not be a valid baseline.")

FINAL VALIDATION
  strongest-64 bin indices                                  : match
  train windows / sensor values / timestamps / class counts : match
  val windows / sensor values / timestamps / class counts   : match
  test windows / sensor values / timestamps / class counts  : match
  seed-42 torch RNG after construction == v16/v18           : match
  fusion backbone is real mamba_ssm                         : match

  alpha_y (training-count weights)            : 1.0892, 0.9675, 0.9829, 0.9790, 0.9908 ✅ unchanged
  control: reused v18 runs (no retraining)    : 0/5 seeds | other seeds: ['regenerated (v18 probabilities not attached)']
  control metrics vs known v18 values         : 42: EXACT, 123: EXACT, 2024: EXACT, 7: EXACT, 99: EXACT | macro-F1 0.9057, 0.8827, 0.8764, 0.9019, 0.8638
  new runs start where the control started    : ✅ all 10
  beta exactly 0.0, lag term 0 at every epoch : ✅ asserted in every run
  confusion matrices printed                  : 10 new runs + 5 contro